In [ ]:
import numpy as np
import os
import time

import rosbag
import yaml

import meshcat
import meshcat.geometry as g
import meshcat.transformations as tf

from tempfile import TemporaryDirectory

from PIL import Image
import io
import rospy
from math_utils import rotation_matrix_to_quaternion, transform_bundletrack_output_to_world
# video resolution
video_resolution=[640, 480]

#
## parameters:
#
cam = 'cam0'
video_name = 'new_toss1'
odom_bag_file = './odom_19.bag'
cam_bag_file = './raw_19.bag'
cam_poses_file = './assets/realsense_pose.yaml'
output_file = './' + video_name + '.mp4'

###################NEW DATASET################
# start time
start_time = rospy.rostime.Time(secs=1667330342, nsecs=959312)  # toss 1
# start_time = rospy.rostime.Time(secs=1667330357, nsecs=453744)  # toss 2
# start_time = rospy.rostime.Time(secs=1667330379, nsecs=657584)  # toss 3
# start_time = rospy.rostime.Time(secs=1667330401, nsecs=359995)  # toss 4
# start_time = rospy.rostime.Time(secs=1667330422, nsecs=301875)  # toss 5
# start_time = rospy.rostime.Time(secs=1667330442, nsecs=284764)  # toss 6
# start_time = rospy.rostime.Time(secs=1667330465, nsecs=216068)  # toss 7
# start_time = rospy.rostime.Time(secs=1667330490, nsecs=318756)  # toss 8

# end time
end_time = rospy.rostime.Time(secs=1667330357, nsecs=453744)  # toss 1
# end_time = rospy.rostime.Time(secs=1667330379, nsecs=657584)  # toss 2
# end_time = rospy.rostime.Time(secs=1667330401, nsecs=359995)  # toss 3
# end_time = rospy.rostime.Time(secs=1667330422, nsecs=301875)  # toss 4
# end_time = rospy.rostime.Time(secs=1667330442, nsecs=284764)  # toss 5
# end_time = rospy.rostime.Time(secs=1667330465, nsecs=216068)  # toss 6
# end_time = rospy.rostime.Time(secs=1667330490, nsecs=318756)  # toss 7
# end_time = rospy.rostime.Time(secs=1667330509, nsecs=187270)  # toss 8

# <------------------------------- Redo toss separation 1/31/23 --------------------------------
# start time
# start_time = rospy.rostime.Time(secs=1655404893, nsecs=899137)  # toss 1

# end time
# end_time = rospy.rostime.Time(secs=1655404908, nsecs=279948)  # toss 1

cam_topic = '/camera/color/image_raw'

intrinsics = [380.2484436035156, 379.8265380859375, 314.2138977050781, 240.59800720214844,] # fx, fy, cx, cy

with open(cam_poses_file, 'r') as stream:
    data_loaded = yaml.safe_load(stream)
print(data_loaded[cam]['pose']['position']['x'])

cam_pos_dict = data_loaded[cam]['pose']['position']
cam_position = [cam_pos_dict['x'], cam_pos_dict['y'], cam_pos_dict['z']]
cam_rot_dict = data_loaded[cam]['pose']['rotation']
cam_orientation = [cam_rot_dict['x'], cam_rot_dict['y'], cam_rot_dict['z']]


# Compute T_WC, transform from world to camera
cam_angle = np.linalg.norm(cam_orientation)
cam_axis = cam_orientation/cam_angle
T_WC = tf.translation_matrix(cam_position) @ tf.quaternion_matrix(tf.quaternion_about_axis(cam_angle, cam_axis))
print(tf.quaternion_matrix(tf.quaternion_about_axis(cam_angle, cam_axis)))

# Given fov_x, fov_y, desire output to be h x w resolution
# set fov=fov_y, screen_h = h
# set screen_w = fov_x/fov_y * w

fx = intrinsics[0]
fy = intrinsics[1]

fov_x = 2*np.arctan(video_resolution[1]/(2*fy))*180/np.pi
fov_y = 2*np.arctan(video_resolution[0]/(2*fx))*180/np.pi


cam_fov = fov_x

print('Extracted field of view: ' + f'{cam_fov:f}')


resolution = video_resolution

vis = meshcat.Visualizer()

In [ ]:
# Create the cubes. Set the opacity of the "real" to > 0 if you want to see it, for invisible
# vis["real_1"].set_object(g.Box([0.1048, 0.1048, 0.1048]),
#                        g.MeshLambertMaterial(
#                              color=0x00ff00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))

vis["real_1"].set_object(g.Box([0.096, 0.061, 0.096]),
                       g.MeshLambertMaterial(
                             color=0x00ff00,
                             reflectivity=0.0,
                             transparent=0,
                             opacity=.4))

vis["real_2"].set_object(g.Box([0.096, 0.061, 0.096]),
                       g.MeshLambertMaterial(
                             color=0xff22dd,
                             reflectivity=0.0,
                             transparent=0,
                             opacity=.4))

# obj = meshcat.geometry.ObjMeshGeometry.from_file("../drake/manipulation/models/franka_description/urdf/panda_arm_hand_wide_finger.urdf")
# finger = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/finger.dae")
# hand = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/hand.dae")
# link_0 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link0.dae")
# link_1 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link1.dae")
# link_2 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link2.dae")
# link_3 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link3.dae")
# link_4 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link4.dae")
# link_5 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link5.dae")
# link_6 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link6.dae")
# link_7 = meshcat.geometry.DaeMeshGeometry.from_file("../franka_description/meshes/visual/link7.dae")
# vis["robot_arm"]["panda_finger"].set_object(finger, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_hand"].set_object(hand, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link0"].set_object(link_0, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link1"].set_object(link_1, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link2"].set_object(link_2, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link3"].set_object(link_3, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link4"].set_object(link_4, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link5"].set_object(link_5, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link6"].set_object(link_6, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))
# vis["robot_arm"]["panda_link7"].set_object(link_7, g.MeshLambertMaterial(
#                              color=0xFFFF00,
#                              reflectivity=0.0,
#                              transparent=0,
#                              opacity=.4))

In [ ]:
def checkStarttime(time1, time2):
    return time1.secs < time2.secs or (
        time1.secs == time2.secs and time1.nsecs < time2.nsecs
    )

def checkEndtime(time1, time2):
    return time1.secs == time2.secs

In [ ]:
raw_bag = rosbag.Bag(cam_bag_file)
info_ = yaml.load(raw_bag._get_yaml_info(), Loader=yaml.FullLoader)
TOPIC_STRING_2 = '/tf'
elbow_2_topic = [topic for topic in info_['topics'] if topic['topic'] == TOPIC_STRING_2][0]

class TFMessage():
    timestamp = None
    transformation = None
    def __init__(self,timestamp, transformation):
        self.timestamp = timestamp
        self.transformation = transformation

def extract_robot_poses(messages, start_time, end_time):
    '''
    Extract joint states from /tf
    '''
    output = []
    for data in messages:
        robot_poses = {}
        tfMessage = None
        if len(data.message.transforms) == 9:
            for msg in data.message.transforms:
                poses = np.zeros((7,1))
                if checkStarttime(msg.header.stamp, start_time):
                    continue
                if checkEndtime(msg.header.stamp, end_time):
                    break
                joint_id = msg.header.frame_id
                if msg.child_frame_id == "panda_leftfinger":
                    joint_id = "panda_lefthand"
                elif msg.child_frame_id == "panda_rightfinger":
                    joint_id = "panda_righthand"
                pose = msg.transform
                pose_pos = np.asarray([pose.translation.x, pose.translation.y, pose.translation.z])
                pose_quat = np.asarray([pose.rotation.w, pose.rotation.x, pose.rotation.y,
                                        pose.rotation.z])
                poses = np.hstack((poses, np.hstack((pose_quat, pose_pos)).T.reshape((-1,1))))
                # poses[:4, i] = pose_quat
                # poses[4:7, i] = pose_pos
                robot_poses[joint_id] = poses[:,1:]
                tfMessage = TFMessage(msg.header.stamp, robot_poses)
                # print(joint_id, robot_poses)
        if tfMessage != None:
            output.append(tfMessage)
    return output

tf_msg = extract_robot_poses(list(raw_bag.read_messages(topics=[TOPIC_STRING_2])), start_time, end_time)

In [ ]:
bag = rosbag.Bag(odom_bag_file)
# get summary info from rosbag as a dictionary
info = yaml.load(bag._get_yaml_info(), Loader=yaml.FullLoader)
TOPIC_STRING_1 = '/tagslam/odom/body_box'

# extract metadata from cube and board topics
elbow_1_topic = [topic for topic in info['topics'] if topic['topic'] == TOPIC_STRING_1][0]

num_msg = elbow_1_topic['messages']

def extract_poses(messages, start_time, end_time):
    # poses = np.zeros((7, len(messages)))
    poses = np.zeros((7,1))
    global odom_timestamps
    odom_timestamps = []
    for i, data in enumerate(messages):
        (_, msg, _) = data
        if checkStarttime(msg.header.stamp, start_time):
            continue
        if checkEndtime(msg.header.stamp, end_time):
            break
        pose = msg.pose.pose
        pose_pos = np.asarray([pose.position.x, pose.position.y, pose.position.z])
        pose_quat = np.asarray([pose.orientation.w, pose.orientation.x, pose.orientation.y,
                                pose.orientation.z])
        # print(np.hstack((pose_quat, pose_pos)).T.shape)
        poses = np.hstack((poses, np.hstack((pose_quat, pose_pos)).T.reshape((-1,1))))
        # poses[:4, i] = pose_quat
        # poses[4:7, i] = pose_pos
        # print(f'poses: {poses}')
        odom_timestamps.append(msg.header.stamp)
    return poses[:,1:]

def get_bundletrack_results():
    """
    State vector is 4 quaternion + 3 xyz position + 3 angular velocity + 3 linear velocity.
    """
    frame_num = len([name for name in os.listdir(DATA_DIR)])
    print("%i frames in total!"%frame_num)
    poses = np.zeros((7, frame_num))
    for frame_id in range(1, frame_num+1):
        pose = np.loadtxt(DATA_DIR + "%04i.txt" % frame_id)#in camera frame
        # For camera extrinsics
        CAMERA_CONFIG = {
            "old": {
                "translation": np.array([[1.14164360], [0.15815239], [0.66422200]]),
                "axis_vec": np.array([-1.57165949, -1.63112887, 1.07928078]),
            },
            "new": {
                "translation": np.array([[1.11076422], [-0.07966290], [0.67947702]]),
                "axis_vec": np.array([-1.61997882, -1.56988553, 0.86362178]),
            },
        }
        pose = transform_bundletrack_output_to_world(
            pose,
            CAMERA_CONFIG["new"]["translation"],
            CAMERA_CONFIG["new"]["axis_vec"],
            DATA_DIR,
            ODOM_FILE_PATH,
        )
        # print(f'pose: {pose}')
        pose_quat = rotation_matrix_to_quaternion(pose)
        # r = R.from_matrix(pose[:3, :3])
        # pose_quat = quaternion_from_matrix(pose)
        # pose_quat_ = r.as_quat()
        # print(f'pose_quat: {pose_quat}, pose_quat_: {pose_quat_}')
        pose_quat = pose_quat.reshape(1, -1)
        pose_pos = np.array(pose[:3, 3])
        poses[:4, frame_id-1] = pose_quat
        poses[4:7, frame_id-1] = pose_pos
    return poses

DATA_DIR = "/home/cnets-vision/mengti_ws/results/poses_1/"
ODOM_FILE_PATH = "/home/cnets-vision/mengti_ws/BundleTrack/Data/YCBINEOAT/contact_nets_new_split/1/annotated_poses/"

bundletrack_poses = get_bundletrack_results()
odom_msg = extract_poses(list(bag.read_messages(topics=[TOPIC_STRING_1])), start_time, end_time)

# print("odom_msg: ", odom_msg.shape)
# print(odom_timestamps)
# if (bundletrack_poses.shape[1]>elbow_1.shape[1]):
#     bundletrack_poses = bundletrack_poses[:, :elbow_1.shape[1]]
# print("bundletrack: ", bundletrack_poses.shape)
bag.close()

In [ ]:
def get_most_recent_odom_idx(tstamp, matched_tstamps):
    '''
    Get the most recent odometry timestamp that is before tstamp.
    tstamp: timestamps of raw images 
    '''
    most_recent = None
    idx = None
    for i, odom_tstamp in enumerate(odom_timestamps):
        if odom_tstamp.secs < tstamp.secs or (odom_tstamp.secs == tstamp.secs and odom_tstamp.nsecs < tstamp.nsecs):
            if most_recent == None or most_recent.secs < odom_tstamp.secs or (most_recent.secs == odom_tstamp.secs and most_recent.nsecs < odom_tstamp.nsecs):
                if odom_tstamp not in matched_tstamps:
                    most_recent = odom_tstamp
                    idx = i
                    matched_tstamps.add(odom_tstamp)
    # print(idx)
    return idx
    

In [ ]:
from cv_bridge import CvBridge

# Load video
raw_bag = rosbag.Bag(cam_bag_file)
cam_messages= list(raw_bag.read_messages(topics=[cam_topic]))
matched_tstamps = set()
num_cam_msg = len(cam_messages)
# print(num_cam_msg)
# extract camera
t_cam = np.zeros(len(cam_messages))
cam_data = []
bridge = CvBridge()
for i, data in enumerate(cam_messages):
    (_, msg, _) = data
    tstamp = msg.header.stamp
    if checkStarttime(tstamp, start_time):
        continue
    if checkEndtime(tstamp, end_time):
        break
    odom_idx = get_most_recent_odom_idx(tstamp, matched_tstamps)
    if odom_idx != None:
        odom_tstamp = odom_timestamps[odom_idx]
        # print('odom: ', odom_tstamp.secs, odom_tstamp.nsecs)
        # print('video: ', tstamp.secs, tstamp.nsecs)
    t_cam[i] = tstamp.secs + tstamp.nsecs * 1e-9
    # cam_data.append(msg.data)
    cv_img = bridge.imgmsg_to_cv2(msg, desired_encoding="passthrough")
    cam_data.append([cv_img, odom_idx, tstamp]) # record tstamps for debugging purposes
print(len(cam_data))
raw_bag.close()

In [ ]:
base_url = "http://127.0.0.1"

meshcat_url = base_url + ":" + vis.url().split(":")[-1]

''' Create precisely-sized iframe with meshcat view; put this in its own Jupyter cell. '''
from IPython.display import HTML
frame_html = """
<div style="height: {height}px; width: {width}px; overflow-x: visible; overflow-y: visible; resize: none">
    <iframe src="{url}" style="width: 100%; height: 100%; border: none"></iframe>
</div>
""".format(url=meshcat_url, width=resolution[0], height=resolution[1])
HTML(frame_html)

In [ ]:
# Frames
# (W) World
# (M) Meshcat
# (C) Camera
# (A) link_1
# (B) link_2

vis["cam"].set_transform(T_WC)
vis["cam_view"].set_transform(T_WC @ tf.translation_matrix([0,0,.05]))

# T_MC, look along z-axis but rotote by 180 degrees
T_MC = tf.translation_matrix([0, 0, -1]) @ tf.rotation_matrix(np.pi, (0,0,1))

T_MW = T_MC @ tf.inverse_matrix(T_WC)

cam = g.PerspectiveCamera(fov=cam_fov, zoom=1, aspect=640/480)
vis["/Cameras/default/rotated"].set_object(cam)

# vis["/Cameras/default/rotated/<object>"].set_property("zoom", 1)
vis["/Cameras/default/rotated/<object>"].set_property("position", [0,0,0])
vis["/Cameras/default"].set_transform(T_MC)



In [ ]:
def get_proper_tf_msg(tfmessages, tstamp):
    most_recent = None
    for tfmessage in tfmessages:
        if tfmessage.timestamp.secs < tstamp.secs or (tfmessage.timestamp.secs == tstamp.secs and tfmessage.timestamp.nsecs < tstamp.nsecs):
            if most_recent == None or most_recent.timestamp.secs < tfmessage.timestamp.secs or (most_recent.timestamp.secs == tfmessage.timestamp.secs \
            and most_recent.timestamp.nsecs < tfmessage.timestamp.nsecs):
                most_recent = tfmessage
    return most_recent

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# view in meshcat save to images
# Turn off background, axes, and grid.
vis['/Background'].set_property("visible", False)
vis['/Grid'].set_property("visible", False)
vis['/Axes'].set_property("visible", False)

with TemporaryDirectory(prefix="ros-process-") as tmpdir:
    print(tmpdir)
    for i, pose in enumerate(bundletrack_poses.T):
        print('Processing frame ' + f'{i:d}' + ' of ' + f'{bundletrack_poses.shape[1]:d}', end='\r')
        im = Image.fromarray(cam_data[i][0]).convert('RGB')
        T_WA = tf.translation_matrix(pose[4:7]) @ tf.quaternion_matrix(pose[:4])
        vis["real_1"].set_transform(T_MW @ T_WA)

        # Visualizing a sphere at (0,0,0)
        # T_WA = tf.translation_matrix(np.array([0,0,0])) @ tf.quaternion_matrix(np.array([0,0,0,1]))
        # vis["real_1"].set_transform(T_MW @ T_WA)
        
        
        # visualizing the ground-truth
        # pose_2 = odom_msg[:,i]
        odom_idx = cam_data[i][1]
        if odom_idx != None:
            pose_2 = odom_msg[:, odom_idx]
            # pose_2 = np.array([0.5276159458071291, -0.5565072250934956, -0.46577238162553075, 0.44156223872028155, 0.7517670087101709, 0.2749021984583954, 0.018160298466467384]).T## For experiment purposes
            
            odom_ = odom_timestamps[odom_idx]
            tstamp = cam_data[i][2]
            # print("odom timestamp: ", odom_.secs, odom_.nsecs, "raw image timestamp:", tstamp.secs, tstamp.nsecs)
            # print(f"odom trans: {pose_2[4:7]}, quat: {pose_2[:4]}")
            T_WB = tf.translation_matrix(pose_2[4:7]) @ tf.quaternion_matrix(pose_2[:4])
            vis["real_2"].set_transform(T_MW @ T_WB)

            # Set manipulator pose
            # robot_pose_ = get_proper_tf_msg(tf_msg, tstamp).transformation
            # pose_link0 = robot_pose_["panda_link0"].reshape((7,))
            # pose_link1 = robot_pose_["panda_link1"].reshape((7,))
            # pose_link2 = robot_pose_["panda_link2"].reshape((7,))
            # pose_link3 = robot_pose_["panda_link3"].reshape((7,))
            # pose_link4 = robot_pose_["panda_link4"].reshape((7,))
            # pose_link5 = robot_pose_["panda_link5"].reshape((7,))
            # pose_link6 = robot_pose_["panda_link6"].reshape((7,))
            # pose_lefthand = robot_pose_["panda_lefthand"].reshape((7,))
            # pose_righthand = robot_pose_["panda_righthand"].reshape((7,))
            # vis["robot_arm"]["panda_link0"].set_transform(tf.translation_matrix(pose_link0[4:7]) @ tf.quaternion_matrix(pose_link0[:4]))
            # vis["robot_arm"]["panda_link1"].set_transform(tf.translation_matrix(pose_link1[4:7]) @ tf.quaternion_matrix(pose_link1[:4]))
            # vis["robot_arm"]["panda_link2"].set_transform(tf.translation_matrix(pose_link2[4:7]) @ tf.quaternion_matrix(pose_link2[:4]))
            # vis["robot_arm"]["panda_link3"].set_transform(tf.translation_matrix(pose_link3[4:7]) @ tf.quaternion_matrix(pose_link3[:4]))
            # vis["robot_arm"]["panda_link4"].set_transform(tf.translation_matrix(pose_link4[4:7]) @ tf.quaternion_matrix(pose_link4[:4]))
            # vis["robot_arm"]["panda_link5"].set_transform(tf.translation_matrix(pose_link5[4:7]) @ tf.quaternion_matrix(pose_link5[:4]))
            # vis["robot_arm"]["panda_link6"].set_transform(tf.translation_matrix(pose_link6[4:7]) @ tf.quaternion_matrix(pose_link6[:4]))
            # vis["robot_arm"]["panda_lefthand"].set_transform(tf.translation_matrix(pose_lefthand[4:7]) @ tf.quaternion_matrix(pose_lefthand[:4]))
            # vis["robot_arm"]["panda_righthand"].set_transform(tf.translation_matrix(pose_righthand[4:7]) @ tf.quaternion_matrix(pose_righthand[:4]))
            # vis["robot_arm"]["panda_hand"]


        mesh_im = vis.get_image()
        # print(mesh_im.size)
        # mesh_im.show()
        im.paste(mesh_im, (0,0), mask = mesh_im)
        if odom_idx != None:
            im.show()
            break
        else:
            continue
        
        # im.save(tmpdir + '/' + f'{i:07d}' + '.png', format="png")
    os.system('ffmpeg -y -r 150 -i ' + tmpdir + '/%07d.png -vcodec libx264 -preset slow -crf 18 ' + output_file)

In [ ]:
from pydrake.common import GetDrakePath
print(GetDrakePath())
%cd /home/cnets-vision/mengti_ws/drake/build
%env PYTHONPATH=${PWD}/install/lib/python3.8/site-packages:${PYTHONPATH}
%cd /home/cnets-vision/mengti_ws/robot_filter
%ls

In [ ]:
from urdf_filter import FrankaPlaybackSim
from pydrake.all import StartMeshcat
from file_utils import import_data

ROOT_DIR = "./dataset/new_split/1/"
POSITION_FILE_PATH = ROOT_DIR + "texts/joint_position.txt"
CAMERA_CONFIG = {
    "old": {
        "translation": np.array([[1.14164360], [0.15815239], [0.66422200]]),
        "axis_vec": np.array([-1.57165949, -1.63112887, 1.07928078]),
    },
    "new": {
        "translation": np.array([[1.11076422], [-0.07966290], [0.67947702]]),
        "axis_vec": np.array([-1.61997882, -1.56988553, 0.86362178]),
    },
}
meshcat = StartMeshcat()
positions = import_data(POSITION_FILE_PATH)

for i, pose in enumerate(bundletrack_poses.T):
    # bundletrack_pose = np.loadtxt(OUTPUT_POSE_DIR + "%04i.txt" % frame_id)
    # pose = transform_bundletrack_output_to_world(
    #     bundletrack_pose,
    #     CAMERA_CONFIG["new"]["translation"],
    #     CAMERA_CONFIG["new"]["axis_vec"],
    #     OUTPUT_POSE_DIR,
    #     ODOM_FILE_PATH,
    # )
    # gt_pose = np.loadtxt(GT_POSE_DIR + "%04i.txt" % frame_id)
    # bundletrack_pose = np.array(
    #     [[1, 0, 0, 0.2], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]
    # )
    # gt_pose = np.array([[1, 0, 0, 0.3], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
    # bundletrack_pose = world_to_camera(gt_pose)
    # gt_pose = camera_to_world(bundletrack_pose)
    odom_idx = cam_data[i][1]
    if odom_idx != None:
        gt_pose = odom_msg[:, odom_idx]
        # pose_2 = np.array([0.5276159458071291, -0.5565072250934956, -0.46577238162553075, 0.44156223872028155, 0.7517670087101709, 0.2749021984583954, 0.018160298466467384]).T## For experiment purposes
        
        odom_ = odom_timestamps[odom_idx]
        tstamp = cam_data[i][2]
        print("odom timestamp: ", odom_.secs, odom_.nsecs, "raw image timestamp:", tstamp.secs, tstamp.nsecs)
        system = FrankaPlaybackSim(
            meshcat,
            positions[i],
            i,
            pose,
            gt_pose,
            translation=CAMERA_CONFIG["new"]["translation"],
            axis_vec=CAMERA_CONFIG["new"]["axis_vec"],
            show=False,
        )